# 🎯 Notebook 4: Optimistic Concurrency Control

Optimistic concurrency control (OCC) takes the opposite approach to pessimistic locking - it **assumes conflicts are rare** and detects them after they occur.

## Learning Objectives

By the end of this notebook, you'll understand:
- How version columns work for concurrency control
- Using existing data as versions (no extra columns)
- The ABA problem and how to avoid it
- When to use optimistic vs pessimistic approaches

## 🎯 How Optimistic Concurrency Works

The pattern is simple:

1. **Read** the current value AND version
2. **Do your work** (no locks held!)
3. **Update** with WHERE clause checking expected version
4. **Retry** if update affects 0 rows (conflict detected)

```sql
-- Alice reads: seats = 1, version = 42
SELECT available_seats, version FROM concerts WHERE id = 1;

-- Alice tries to update
UPDATE concerts 
SET available_seats = 0, version = version + 1
WHERE id = 1 AND version = 42;  -- Expected version

-- If 0 rows affected → someone else modified it!
```

---

🔍 **Open Adminer** at http://localhost:8080 → Watch the `version` column change in the `concerts` table!

In [ ]:
import psycopg2
from concurrent.futures import ThreadPoolExecutor
import time
from threading import Lock

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "contention_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

event_log = []
log_lock = Lock()

def log_event(user: str, event: str):
    with log_lock:
        timestamp = time.time()
        event_log.append({"time": timestamp, "user": user, "event": event})

try:
    conn = get_connection()
    print("✅ Connected to PostgreSQL")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
def reset_concert(seats: int = 1):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE concerts SET available_seats = %s, version = 1 WHERE id = 1", (seats,))
    cursor.execute("DELETE FROM tickets WHERE concert_id = 1")
    conn.commit()
    conn.close()

def get_concert_status():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT available_seats, version FROM concerts WHERE id = 1")
    row = cursor.fetchone()
    cursor.execute("SELECT COUNT(*) FROM tickets WHERE concert_id = 1")
    tickets = cursor.fetchone()[0]
    conn.close()
    return {"seats": row[0], "version": row[1], "tickets": tickets}

reset_concert(1)
status = get_concert_status()
print(f"🎫 Concert status: {status['seats']} seats, version {status['version']}")

## 🎫 Optimistic Ticket Purchase

In [ ]:
def buy_ticket_optimistic(user_id: str, max_retries: int = 3) -> dict:
    for attempt in range(max_retries):
        conn = get_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute(
                "SELECT available_seats, version FROM concerts WHERE id = 1"
            )
            seats, version = cursor.fetchone()
            
            log_event(user_id, f"Read: {seats} seats, version {version}")
            
            if seats < 1:
                log_event(user_id, "❌ Sold out")
                return {"success": False, "user": user_id, "reason": "sold_out"}
            
            time.sleep(0.01)
            
            cursor.execute(
                "UPDATE concerts "
                "SET available_seats = available_seats - 1, version = version + 1 "
                "WHERE id = 1 AND version = %s",
                (version,)
            )
            
            rows_affected = cursor.rowcount
            
            if rows_affected == 0:
                conn.rollback()
                log_event(user_id, f"🔄 Conflict detected (attempt {attempt + 1}), retrying...")
                conn.close()
                continue
            
            cursor.execute(
                "INSERT INTO tickets (concert_id, user_id, seat_number, purchase_price) "
                "VALUES (1, %s, 'A1', 150.00)",
                (user_id,)
            )
            conn.commit()
            log_event(user_id, "✅ Purchase successful!")
            return {"success": True, "user": user_id, "attempts": attempt + 1}
            
        except Exception as e:
            conn.rollback()
            log_event(user_id, f"ERROR: {e}")
            return {"success": False, "user": user_id, "reason": str(e)}
        finally:
            conn.close()
    
    log_event(user_id, "❌ Max retries exceeded")
    return {"success": False, "user": user_id, "reason": "max_retries"}

print("✅ Optimistic concurrency function created")

In [ ]:
print("🎯 Testing Optimistic Concurrency")
print("=" * 60)

reset_concert(1)
event_log.clear()

print("Starting state: 1 seat, version 1")
print("Alice and Bob both try to buy...\n")

with ThreadPoolExecutor(max_workers=2) as executor:
    f_alice = executor.submit(buy_ticket_optimistic, "Alice")
    f_bob = executor.submit(buy_ticket_optimistic, "Bob")
    
    r_alice = f_alice.result()
    r_bob = f_bob.result()

print("📋 Event Log:")
for event in sorted(event_log, key=lambda x: x["time"]):
    print(f"   [{event['user']:5s}] {event['event']}")

print(f"\n📊 Results:")
print(f"   Alice: {r_alice}")
print(f"   Bob:   {r_bob}")

status = get_concert_status()
print(f"\n🎫 Final: {status['seats']} seats, version {status['version']}, {status['tickets']} tickets")

if (r_alice['success'] != r_bob['success']) and status['seats'] == 0:
    print("\n✅ SUCCESS! Optimistic concurrency prevented the race condition!")

## 📊 Stress Test

In [ ]:
def stress_test_optimistic(num_users: int, available_seats: int):
    reset_concert(available_seats)
    
    def try_buy(user_num):
        return buy_ticket_optimistic(f"User{user_num}")
    
    with ThreadPoolExecutor(max_workers=num_users) as executor:
        futures = [executor.submit(try_buy, i) for i in range(num_users)]
        results = [f.result() for f in futures]
    
    successful = sum(1 for r in results if r["success"])
    total_attempts = sum(r.get("attempts", 1) for r in results if r["success"])
    status = get_concert_status()
    
    return {
        "users": num_users,
        "seats": available_seats,
        "purchased": successful,
        "final_seats": status["seats"],
        "total_attempts": total_attempts,
        "correct": successful == available_seats and status["seats"] == 0
    }

print("🧪 Stress Testing Optimistic Concurrency")
print("=" * 70)
print()

event_log.clear()

test_cases = [
    (5, 3),
    (10, 5),
    (20, 10),
    (50, 20),
]

all_correct = True
for num_users, seats in test_cases:
    result = stress_test_optimistic(num_users, seats)
    status = "✅" if result["correct"] else "❌"
    if not result["correct"]:
        all_correct = False
    retries = result["total_attempts"] - result["purchased"]
    print(f"Users: {result['users']:3d} | Seats: {result['seats']:3d} | "
          f"Sold: {result['purchased']:3d} | Retries: {retries:3d} | {status}")

print()
if all_correct:
    print("🎉 All tests passed!")
    print("💡 Notice: More users = More retries (but still correct!)")

## 🏷️ Using Existing Data as Version

You don't always need a separate `version` column. You can use existing data that naturally changes:

In [ ]:
def buy_ticket_using_seats_as_version(user_id: str, max_retries: int = 3) -> dict:
    for attempt in range(max_retries):
        conn = get_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute(
                "SELECT available_seats FROM concerts WHERE id = 1"
            )
            seats = cursor.fetchone()[0]
            
            if seats < 1:
                return {"success": False, "user": user_id, "reason": "sold_out"}
            
            time.sleep(0.01)
            
            cursor.execute(
                "UPDATE concerts "
                "SET available_seats = available_seats - 1 "
                "WHERE id = 1 AND available_seats = %s",
                (seats,)
            )
            
            if cursor.rowcount == 0:
                conn.rollback()
                conn.close()
                continue
            
            cursor.execute(
                "INSERT INTO tickets (concert_id, user_id, seat_number, purchase_price) "
                "VALUES (1, %s, 'A1', 150.00)",
                (user_id,)
            )
            conn.commit()
            return {"success": True, "user": user_id, "attempts": attempt + 1}
            
        except Exception as e:
            conn.rollback()
            return {"success": False, "user": user_id, "reason": str(e)}
        finally:
            conn.close()
    
    return {"success": False, "user": user_id, "reason": "max_retries"}

print("🏷️ Using available_seats as implicit version")
print("=" * 50)

reset_concert(1)

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(buy_ticket_using_seats_as_version, "Alice")
    f2 = executor.submit(buy_ticket_using_seats_as_version, "Bob")
    
    r1 = f1.result()
    r2 = f2.result()

print(f"Alice: {r1}")
print(f"Bob:   {r2}")

status = get_concert_status()
print(f"\nFinal: {status['seats']} seats, {status['tickets']} tickets")
print("\n💡 No extra version column needed! The seat count IS the version.")

## 🎰 Real-World: Auction Bidding

Auctions are a perfect use case for OCC - use the current bid as the version!

In [ ]:
def place_bid(user_id: str, item_id: int, bid_amount: float, max_retries: int = 3) -> dict:
    for attempt in range(max_retries):
        conn = get_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute(
                "SELECT current_bid, bid_count, status FROM auction_items WHERE id = %s",
                (item_id,)
            )
            current_bid, bid_count, status = cursor.fetchone()
            
            if status != 'active':
                return {"success": False, "reason": "auction_ended"}
            
            if bid_amount <= current_bid:
                return {"success": False, "reason": "bid_too_low", "current_bid": float(current_bid)}
            
            cursor.execute(
                "UPDATE auction_items "
                "SET current_bid = %s, highest_bidder = %s, bid_count = bid_count + 1 "
                "WHERE id = %s AND bid_count = %s",
                (bid_amount, user_id, item_id, bid_count)
            )
            
            if cursor.rowcount == 0:
                conn.rollback()
                conn.close()
                continue
            
            cursor.execute(
                "INSERT INTO bids (auction_item_id, user_id, bid_amount) VALUES (%s, %s, %s)",
                (item_id, user_id, bid_amount)
            )
            conn.commit()
            return {"success": True, "user": user_id, "bid": bid_amount, "attempts": attempt + 1}
            
        except Exception as e:
            conn.rollback()
            return {"success": False, "reason": str(e)}
        finally:
            conn.close()
    
    return {"success": False, "reason": "max_retries"}

def reset_auction(item_id: int = 1, starting_bid: float = 100):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE auction_items SET current_bid = %s, highest_bidder = NULL, "
        "bid_count = 0, status = 'active' WHERE id = %s",
        (starting_bid, item_id)
    )
    cursor.execute("DELETE FROM bids WHERE auction_item_id = %s", (item_id,))
    conn.commit()
    conn.close()

print("🎰 Auction Bidding Simulation")
print("=" * 50)

reset_auction(1, 100)

def bid_war(user_id: str, bids: list):
    results = []
    for bid in bids:
        result = place_bid(user_id, 1, bid)
        results.append(result)
        time.sleep(0.01)
    return results

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(bid_war, "Alice", [150, 200, 250])
    f2 = executor.submit(bid_war, "Bob", [160, 220, 280])
    
    alice_results = f1.result()
    bob_results = f2.result()

print("\nAlice's bids:")
for r in alice_results:
    status = "✅" if r.get("success") else "❌"
    print(f"   {status} {r}")

print("\nBob's bids:")
for r in bob_results:
    status = "✅" if r.get("success") else "❌"
    print(f"   {status} {r}")

conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT current_bid, highest_bidder, bid_count FROM auction_items WHERE id = 1")
final = cursor.fetchone()
conn.close()

print(f"\n🏆 Final: ${final[0]} by {final[1]} ({final[2]} total bids)")

## ⚠️ The ABA Problem

The ABA problem occurs when a value changes from A → B → A. Your version check sees "A" and thinks nothing changed, but important transitions happened.

```
Thread 1 reads: balance = $100
Thread 2: withdraws $50 (balance = $50)
Thread 2: deposits $50 (balance = $100)
Thread 1: CAS(100, 90) succeeds!  ← Thinks nothing happened
```

In [ ]:
print("⚠️ ABA Problem Example")
print("=" * 50)
print()
print("Scenario: Yelp restaurant ratings")
print("Restaurant has 4.0 stars with 100 reviews")
print()
print("Timeline:")
print("1. Alice reads rating: 4.0 stars")
print("2. Bob submits 5-star review, rating becomes 4.01")
print("3. Carol submits 3-star review, rating becomes... 4.0!")
print("4. Alice tries UPDATE WHERE rating = 4.0")
print("   → Succeeds! But missed 2 reviews!")
print()
print("🔑 Solution: Use a value that ALWAYS changes (like review_count)")
print("   UPDATE ... WHERE review_count = 100")
print("   → This will fail correctly because count is now 102")

## ⚖️ Optimistic vs Pessimistic

| Factor | Optimistic | Pessimistic |
|--------|------------|-------------|
| Assumption | Conflicts are rare | Conflicts are common |
| When conflict detected | After work done | Before work starts |
| Lock duration | None | Until commit |
| Throughput (low contention) | ✅ Higher | Lower |
| Throughput (high contention) | Lower (retries) | ✅ Predictable |
| Wasted work | Yes (on conflicts) | No |
| Deadlocks | Never | Possible |
| Complexity | Medium (retry logic) | Low |

In [ ]:
print("📊 When to Use Each Approach")
print("=" * 60)
print()
print("USE OPTIMISTIC when:")
print("   ✅ Conflicts are rare (< 10% of transactions)")
print("   ✅ Transactions are short")
print("   ✅ Retrying is cheap")
print("   ✅ You can't afford lock waiting time")
print("   Examples: Auction bids, inventory updates, ratings")
print()
print("USE PESSIMISTIC when:")
print("   ✅ Conflicts are common (hot resources)")
print("   ✅ Work before commit is expensive")
print("   ✅ Predictable performance matters")
print("   ✅ You need to avoid retries")
print("   Examples: Concert tickets, bank transfers, seat selection")

## 🧪 Quick Quiz

1. **What happens when an optimistic update affects 0 rows?**

2. **Why is bid_count better than current_bid as a version for auctions?**

3. **When would optimistic perform WORSE than pessimistic?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. When 0 rows affected:")
print("   - Conflict detected! Someone else modified the row.")
print("   - Application should RETRY with fresh data.")
print("   - No error thrown - you check rowcount yourself.")
print()
print("2. bid_count vs current_bid:")
print("   - bid_count ALWAYS increases (monotonic)")
print("   - current_bid could theoretically have ABA issues")
print("   - (e.g., two identical bids in different currencies)")
print("   - bid_count is the 'version' that can never repeat")
print()
print("3. Optimistic performs worse when:")
print("   - High contention (many conflicts = many retries)")
print("   - Expensive work before commit (wasted on retry)")
print("   - Example: 100 users competing for 1 ticket")

## 📚 Summary

### What We Learned

1. **Optimistic concurrency** assumes conflicts are rare
2. **Version columns** or **existing data** detect changes
3. **0 rows affected** = conflict detected, retry
4. **ABA problem** - use monotonically increasing values
5. **Choose based on contention level**

### Key Insight

> Optimistic concurrency trades potential retries for no blocking. Under low contention, this is a huge win. Under high contention, retries become expensive and pessimistic locking is better.

### Next Up: Distributed Coordination

In the next notebook, we'll explore what happens when you need to coordinate across **multiple databases** - Two-Phase Commit, Sagas, and Distributed Locks!